# Studio 2 — Klasifikasi Sampah (Colab)
Jalankan dari atas ke bawah. **Runtime > Change runtime type > T4 GPU** dulu.

Dataset tidak boleh diunggah ke repo publik / Kaggle / Hugging Face. Di notebook ini dataset diambil dari **Google Drive pribadi** kamu.

## 1. Setup

In [ ]:
!nvidia-smi -L
REPO_URL = "https://github.com/<USERNAME>/<NAMA-REPO>.git"   # <- ganti dengan repo kelompok
!git clone {REPO_URL} repo
%cd repo
!pip install -q -r requirements.txt

## 2. Siapkan dataset
Salin folder dataset dari Drive (taruh dulu di `MyDrive/studio2/`, berisi `train/` atau `train.zip`).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

SRC = "/content/drive/MyDrive/studio2"   # <- sesuaikan
import os
if os.path.exists(f"{SRC}/train.zip"):
    !unzip -q -o {SRC}/train.zip -d /content/data
    DATA = "/content/data/train"
else:
    !mkdir -p /content/data && cp -r {SRC}/train /content/data/
    DATA = "/content/data/train"
!ls {DATA}   # harus ada 5 folder kelas

## 3. Baseline dulu (amankan Preview 1)
ResNet-18, full fine-tuning, `--export` langsung menaruh hasil di `submission/`.

In [ ]:
!python train.py --data {DATA} --split-dir /content/split --backbone resnet18 --strategy full --epochs 8 --lr 3e-4 --export
!python evaluate.py --submission ./submission --data /content/split/my_val

## 4. Eksperimen: backbone x strategi
Semua hasil masuk `runs/results.csv`. Kurangi daftar kalau waktu GPU terbatas
(mulai dari `convnext_tiny` dan `resnet18` saja).

In [ ]:
BACKBONES = ["resnet18", "resnet50", "mobilenet_v3_large", "efficientnet_b0", "convnext_tiny"]
STRATEGIES = ["linear_probe", "gradual", "full"]
EPOCHS = 12

for b in BACKBONES:
    for s in STRATEGIES:
        lr = "3e-4" if s == "full" else "1e-3"
        !python train.py --data {DATA} --split-dir /content/split --backbone {b} --strategy {s} --epochs {EPOCHS} --lr {lr}

In [ ]:
import pandas as pd
df = pd.read_csv("runs/results.csv")
cols = ["tag","backbone","strategy","img_size","params_total","params_trained_final","macro_f1","accuracy","seconds"]
df[cols].sort_values("macro_f1", ascending=False)

## 5. Ekspor model terbaik ke `submission/`
Ganti `TAG` dengan nama run terbaik dari tabel di atas, lalu uji dengan `evaluate.py` resmi.

In [ ]:
TAG = "convnext_tiny_gradual_224_s42"   # <- ganti
!cp runs/{TAG}/model.pth submission/model.pth
!cp runs/{TAG}/config.json submission/config.json
!python evaluate.py --submission ./submission --data /content/split/my_val

## 6. Unduh hasil lalu commit dari laptop
Unduh `model.pth`, `config.json`, dan `results.csv`, taruh di repo lokal, lalu `git add` / `git commit` / `git push`.
Bobot > 100 MB harus lewat GitHub Release atau Git LFS (tulis tautannya di README).

In [ ]:
from google.colab import files
for f in ["submission/model.pth", "submission/config.json", "runs/results.csv"]:
    files.download(f)

## (Opsional) Model final: latih ulang di seluruh data
Setelah konfigurasi terbaik terkunci, latih ulang di train + my_val. Skor yang tercetak optimistis
(my_val ikut dilatih); simpan epoch terakhir. Bandingkan dulu dengan model tanpa `--full-data` lewat skor preview.

In [ ]:
!python train.py --data {DATA} --split-dir /content/split --backbone convnext_tiny --strategy gradual --epochs 12 --full-data --tag final_full --export